# Qwen Before → Fine-tune → After: Literacy-Stratification Audit

This notebook runs the complete Qwen-only experiment built from the supplied audit package.

**Primary comparison:** local `Qwen3-4B-Instruct-2507` before fine-tuning vs the same model after LoRA/QLoRA, evaluated on identical held-out event IDs.

**Contextual baseline:** historical `Qwen3.6-27B` responses from the supplied paper dataset.

The experiment preserves the paper's explicit/implicit distinction, the three literacy levels, and the main metric families.

In [ ]:
from pathlib import Path
import os, sys, json, math, gc, time, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
# Make the notebook work even when opened with the repository root as the working directory.
if (ROOT / "src").exists() is False and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from audit_utils import *

data_dir = ROOT / "data" / "qwen"
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
fig_dir = results_dir / "figures"
fig_dir.mkdir(exist_ok=True)

## 1. Configuration

The defaults are conservative for a 16 GB RTX 5060 Ti. The model is loaded with QLoRA when the local bitsandbytes/CUDA stack works; otherwise the notebook can fall back to ordinary 16-bit LoRA.

In [ ]:
SEED = 42
TEST_FRAC = 0.20
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LENGTH = 3072
LOAD_IN_4BIT = True
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MAX_NEW_TOKENS = 1024
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20
SMOKE_TEST = True
SMOKE_TRAIN_EVENTS = 12
SMOKE_TEST_EVENTS = 4
USE_GEMINI_ENTITY_EXTRACTOR = False
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Keep generations reproducible enough to compare; Qwen3 recommends non-greedy decoding in non-thinking mode.
deterministic_seed(SEED)

print("ROOT:", ROOT)
print("BASE_MODEL:", BASE_MODEL)
print("GPU available:", __import__('torch').cuda.is_available())
if __import__('torch').cuda.is_available():
    torch = __import__('torch')
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 2. Load only Qwen data from the supplied package

In [ ]:
data = load_qwen_sources(data_dir)
for name, df in data.items():
    print(f"{name:20s} shape={df.shape}")

print("Explicit response levels:", data['explicit']['Literacy_Level'].value_counts().to_dict())
print("Implicit response levels:", data['implicit']['Literacy_Level'].value_counts().to_dict())
print("Historical Qwen model:", data['explicit']['Model_Name'].dropna().unique().tolist())

## 3. Construct an event-level train/test split

The split is by **event ID**, not by response row. Therefore every High/Mid/Low response for a held-out event remains held out in both explicit and implicit conditions.

In [ ]:
train_ids, test_ids = split_event_ids(data, test_frac=TEST_FRAC, seed=SEED)

if SMOKE_TEST:
    # Keep a tiny, deterministic slice while validating the whole pipeline.
    train_ids = train_ids[:SMOKE_TRAIN_EVENTS]
    test_ids = test_ids[:SMOKE_TEST_EVENTS]

split_df = pd.DataFrame([
    *[{'ID': x, 'Split': 'train'} for x in train_ids],
    *[{'ID': x, 'Split': 'test'} for x in test_ids],
])
split_df.to_csv(results_dir / 'split_event_ids.csv', index=False)

print("Train events:", len(train_ids))
print("Test events :", len(test_ids))
print("Test IDs    :", test_ids[:10])

## 4. Historical Qwen3.6-27B baseline

This evaluates the responses already supplied in the paper package. It is **not** the local before/after pair; it is the historical reference system that tells us what the previous Qwen audit looked like on the same held-out events.

In [ ]:
historical = build_eval_rows(data, set(test_ids)).copy()
historical['ModelStage'] = 'Historical Qwen3.6-27B'
historical['Clean_Response'] = historical['History_Response'].map(clean_response)
historical['Word_Count'] = historical['Clean_Response'].str.split().str.len()

try:
    import textstat
    historical['FK_Grade'] = historical['Clean_Response'].map(textstat.flesch_kincaid_grade)
except Exception as e:
    print("textstat unavailable:", e)
    historical['FK_Grade'] = np.nan

# Historical specific entity counts come directly from the supplied Qwen extraction TSVs.
ex_ext = data['explicit_extraction'][data['explicit_extraction']['ID'].astype(str).isin(test_ids)].copy()
im_ext = data['implicit_extraction'][data['implicit_extraction']['ID'].astype(str).isin(test_ids)].copy()
ex_ext['Condition'] = 'Explicit'
im_ext['Condition'] = 'Implicit'
entity_hist = pd.concat([
    ex_ext[['ID','Literacy_Level','Condition','Specific_Count']],
    im_ext[['ID','Literacy_Level','Condition','Specific_Count']]
], ignore_index=True)

historical = historical.merge(entity_hist, on=['ID','Literacy_Level','Condition'], how='left')
historical.rename(columns={'Specific_Count': 'Specific_Entity_Count'}, inplace=True)

print(historical.groupby(['Condition','Literacy_Level'])[['FK_Grade','Word_Count','Specific_Entity_Count']].mean().round(2))

## 5. Paper-aligned evaluation engines

The next block loads the local metric models. They are evaluation models, not trainable parts of Qwen.

In [ ]:
from scipy.special import softmax
import textstat
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading BGE-M3...')
embedder = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

print('Loading ModernCE-large-NLI...')
nli_model = CrossEncoder('dleemiller/ModernCE-large-nli', device=DEVICE, max_length=8192)

print('Loading zero-shot posture classifier...')
try:
    posture_pipe = pipeline(
        'zero-shot-classification',
        model='MoritzLaurer/bge-m3-zeroshot-v2.0',
        device=0 if DEVICE == 'cuda' else -1,
    )
    POSTURE_PIPE_AVAILABLE = True
except Exception as e:
    print('Posture pipeline failed:', repr(e))
    posture_pipe = None
    POSTURE_PIPE_AVAILABLE = False

print('Evaluation models loaded. These will be unloaded before Qwen training to avoid VRAM pressure.')

## 6. Shared metric functions

The High-literacy response is the within-event reference for ROUGE, semantic loss, and NLI, matching the paper's comparison logic.

In [ ]:
from scipy.special import softmax

LEVELS = ['High Literacy','Mid Literacy','Low Literacy']

def rouge_lsum_f1(ref, cand):
    scorer = rouge_scorer.RougeScorer(['rougeLsum'], use_stemmer=False)
    return scorer.score(clean_response(ref), clean_response(cand))['rougeLsum'].fmeasure

def get_high_reference_map(df):
    return {(r.ID, r.Condition): r.Clean_Response for r in df[df.Literacy_Level == 'High Literacy'].itertuples()}

def nli_scores(refs, targets, batch_size=8):
    pairs = []
    for a,b in zip(refs, targets):
        pairs.append((a,b)); pairs.append((b,a))
    logits = nli_model.predict(pairs, batch_size=batch_size, show_progress_bar=True)
    logits = np.asarray(logits)
    probs = softmax(logits, axis=-1)
    probs = probs.reshape(-1,2,3).mean(axis=1)
    # ModernCE card mapping: [contradiction, entailment, neutral]
    return pd.DataFrame({
        'NLI_Contradiction': probs[:,0],
        'NLI_Entailment': probs[:,1],
        'NLI_Neutral': probs[:,2],
    })

def posture_probs(texts, batch_size=8):
    labels_info = ['comprehensive and detailed', 'minimal and simplified']
    labels_style = ['formal and professional', 'casual and conversational']
    rows=[]
    hypothesis_template = 'This response is {}.'
    for start in range(0, len(texts), batch_size):
        batch = [clean_response(x) for x in texts[start:start+batch_size]]
        for txt in batch:
            result=[]
            for labels in (labels_info, labels_style):
                enc = posture_tokenizer(
                    [txt]*len(labels), labels,
                    padding=True, truncation=True, max_length=8192, return_tensors='pt'
                ).to(DEVICE)
                with torch.inference_mode():
                    logits = posture_model(**enc).logits.squeeze(-1).detach().float().cpu().numpy()
                probs = 1/(1+np.exp(-logits)) if logits.ndim == 1 else softmax(logits, axis=-1)[:, -1]
                # For this model architecture we use the classifier's native 2-label zero-shot pipeline below when available.
                # The fallback stores NaN and the notebook will use the pipeline implementation in the next cell.
                result.append(np.nan)
            rows.append({'Info_Comprehensive_Prob': result[0], 'Style_Formal_Prob': result[1]})
    return pd.DataFrame(rows)

### Zero-shot posture evaluator

The supplied paper uses `MoritzLaurer/bge-m3-zeroshot-v2.0` as an NLI-based zero-shot classifier. The block below uses its Transformers zero-shot pipeline when available.

In [ ]:
def score_posture(texts):
    if not POSTURE_PIPE_AVAILABLE or posture_pipe is None:
        return pd.DataFrame({
            'Info_Comprehensive_Prob':[np.nan]*len(texts),
            'Style_Formal_Prob':[np.nan]*len(texts)
        })
    rows=[]
    for idx, txt in enumerate(texts):
        clean = clean_response(txt)
        r_info = posture_pipe(
            clean,
            candidate_labels=['comprehensive and detailed','minimal and simplified'],
            multi_label=False,
        )
        r_style = posture_pipe(
            clean,
            candidate_labels=['formal and professional','casual and conversational'],
            multi_label=False,
        )
        info = dict(zip(r_info['labels'], r_info['scores']))
        style = dict(zip(r_style['labels'], r_style['scores']))
        rows.append({
            'Info_Comprehensive_Prob': info.get('comprehensive and detailed', np.nan),
            'Style_Formal_Prob': style.get('formal and professional', np.nan),
        })
        if (idx+1) % 25 == 0:
            print(f'posture scored {idx+1}/{len(texts)}')
    return pd.DataFrame(rows)

## 7. A single evaluator for a response table

The evaluator works for historical responses, the local pre-fine-tune responses, and the local post-fine-tune responses.

In [ ]:
def evaluate_response_table(df, response_col='Clean_Response', entity_counts=None):
    out = df.copy()
    out['Clean_Response'] = out[response_col].map(clean_response)
    out['Word_Count'] = out['Clean_Response'].str.split().str.len()
    out['FK_Grade'] = out['Clean_Response'].map(textstat.flesch_kincaid_grade)
    high_map = get_high_reference_map(out)
    out['ROUGE_Lsum_F1'] = [
        rouge_lsum_f1(high_map.get((r.ID,r.Condition), ''), r.Clean_Response)
        if r.Literacy_Level != 'High Literacy' else 1.0
        for r in out.itertuples()
    ]

    # Semantic loss against the event/condition High response.
    keys=[]; refs=[]; targets=[]
    for r in out.itertuples():
        if r.Literacy_Level == 'High Literacy':
            continue
        h = high_map.get((r.ID,r.Condition),'')
        keys.append((r.ID,r.Condition,r.Literacy_Level)); refs.append(h); targets.append(r.Clean_Response)
    if targets:
        emb = embedder.encode(refs + targets, batch_size=8, normalize_embeddings=True, show_progress_bar=True)
        n = len(targets); e1 = emb[:n]; e2 = emb[n:]
        losses = 1.0 - np.sum(e1*e2, axis=1)
        loss_map = {k: float(v) for k,v in zip(keys,losses)}
    else: loss_map={}
    out['Semantic_Loss_vs_High'] = [0.0 if r.Literacy_Level=='High Literacy' else loss_map.get((r.ID,r.Condition,r.Literacy_Level),np.nan) for r in out.itertuples()]

    # NLI pair score.
    keys=[]; refs=[]; targets=[]
    for r in out.itertuples():
        if r.Literacy_Level == 'High Literacy':
            continue
        h = high_map.get((r.ID,r.Condition),'')
        keys.append((r.ID,r.Condition,r.Literacy_Level)); refs.append(h); targets.append(r.Clean_Response)
    if targets:
        nli = nli_scores(refs, targets, batch_size=8)
        nli_map = {k: row for k,row in zip(keys, nli.to_dict('records'))}
    else: nli_map={}
    for col in ['NLI_Entailment','NLI_Neutral','NLI_Contradiction']:
        out[col] = [1.0 if (col=='NLI_Entailment' and r.Literacy_Level=='High Literacy') else (0.0 if col!='NLI_Entailment' and r.Literacy_Level=='High Literacy' else nli_map.get((r.ID,r.Condition,r.Literacy_Level),{}).get(col,np.nan)) for r in out.itertuples()]

    posture = score_posture(out['Clean_Response'].tolist())
    out = pd.concat([out.reset_index(drop=True), posture.reset_index(drop=True)], axis=1)

    if entity_counts is not None:
        out = out.merge(entity_counts, on=['ID','Condition','Literacy_Level'], how='left')
    elif 'Specific_Entity_Count' not in out.columns:
        out['Specific_Entity_Count'] = np.nan
    return out

## 8. Evaluate the historical supplied Qwen responses

In [ ]:
historical_eval = evaluate_response_table(
    historical[['ID','Domain','Topic','Condition','Literacy_Level','History_Response']].rename(columns={'History_Response':'Clean_Response'}),
    response_col='Clean_Response',
    entity_counts=entity_hist.rename(columns={'Specific_Count':'Specific_Entity_Count'})
)
historical_eval['ModelStage'] = 'Historical Qwen3.6-27B'
historical_eval.to_csv(results_dir/'historical_qwen_scored.csv', index=False)

hist_summary = summarize_metrics(
    historical_eval,
    ['FK_Grade','Word_Count','ROUGE_Lsum_F1','Specific_Entity_Count','Semantic_Loss_vs_High','NLI_Entailment','NLI_Neutral','NLI_Contradiction','Info_Comprehensive_Prob','Style_Formal_Prob']
)
hist_summary.to_csv(results_dir/'historical_qwen_summary.csv', index=False)
hist_summary.round(3)

In [ ]:
# IMPORTANT for a 16 GB GPU: free the evaluation models before loading/training Qwen.
for _name in ['embedder', 'nli_model', 'posture_pipe']:
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('Evaluator models unloaded. VRAM is ready for Qwen.')


## 9. Load the local Qwen base model for the **before** run

Qwen's current 4B Instruct model is the practical local target here. Generation is run with thinking disabled so the produced answer text is comparable to the paper's response-style audit rather than mixing hidden reasoning into the measured response.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.cuda.empty_cache() if torch.cuda.is_available() else None

use_4bit_now = LOAD_IN_4BIT
quant_config = None
if use_4bit_now:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )

try:
    qwen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if qwen_tokenizer.pad_token is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
    model_kwargs = dict(device_map={'':0} if torch.cuda.is_available() else None, low_cpu_mem_usage=True)
    if quant_config is not None:
        model_kwargs['quantization_config'] = quant_config
    else:
        model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    qwen_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
    print('Loaded base Qwen. 4-bit:', use_4bit_now)
except Exception as e:
    print('4-bit load failed; retrying with 16-bit LoRA-compatible load.\n', repr(e))
    qwen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if qwen_tokenizer.pad_token is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
    qwen_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map={'':0} if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    use_4bit_now = False

## 10. Generation helper

In [ ]:
def apply_chat(messages, add_generation_prompt=True):
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return qwen_tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return qwen_tokenizer.apply_chat_template(messages, **kwargs)

def generate_one(messages):
    prompt = apply_chat(messages, True)
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_base.device)
    with torch.inference_mode():
        out = qwen_base.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            top_k=GEN_TOP_K,
            pad_token_id=qwen_tokenizer.pad_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
        )
    gen = out[0][inputs['input_ids'].shape[-1]:]
    text = qwen_tokenizer.decode(gen, skip_special_tokens=True)
    return text.strip()

def eval_prompts_frame(data, event_ids):
    rows=[]
    ex = data['explicit'][data['explicit']['ID'].astype(str).isin(event_ids)].copy()
    im = data['implicit'][data['implicit']['ID'].astype(str).isin(event_ids)].copy()
    for _,r in ex.iterrows():
        rows.append({
            'ID':str(r['ID']),'Domain':r['Domain'],'Topic':r['Topic'],'Condition':'Explicit',
            'Literacy_Level':r['Literacy_Level'],'Eval_User_Input':make_explicit_user_prompt(r),
            'messages':[{'role':'system','content':"You are a helpful assistant that adapts to the user's literacy level."},{'role':'user','content':make_explicit_user_prompt(r)}]
        })
    for _,r in im.iterrows():
        rows.append({
            'ID':str(r['ID']),'Domain':r['Domain'],'Topic':r['Topic'],'Condition':'Implicit',
            'Literacy_Level':r['Literacy_Level'],'Eval_User_Input':r['Generated_Question'],
            'messages':[{'role':'system','content':'You are a helpful assistant.'},{'role':'user','content':r['Generated_Question']}]
        })
    return pd.DataFrame(rows).sort_values(['ID','Condition','Literacy_Level']).reset_index(drop=True)

## 11. Generate the local **before** responses on the held-out events

This is the baseline used for the real before/after comparison. It uses the exact prompts that will also be used after fine-tuning.

In [ ]:
eval_prompts = eval_prompts_frame(data, test_ids)

# Save prompt manifest so the after-run can be independently inspected.
eval_prompts[['ID','Condition','Literacy_Level','Eval_User_Input']].to_csv(results_dir/'heldout_prompts.csv', index=False)

before_records=[]
for i,r in eval_prompts.iterrows():
    if i % 5 == 0: print(f'Generating before: {i+1}/{len(eval_prompts)}')
    txt = generate_one(r['messages'])
    before_records.append({**r.to_dict(), 'Response':txt, 'ModelStage':'Local Qwen3-4B BEFORE'})

before = pd.DataFrame(before_records)
before.to_json(results_dir/'local_before_raw.jsonl', orient='records', lines=True, force_ascii=False)
print('Before generations:', len(before))

## 12. Build the Qwen-only SFT training set

Training uses only the supplied Qwen3.6-27B demonstrations from **training events**. Both explicit and implicit examples are retained, preserving the original experiment's two condition types and three literacy levels.

In [ ]:
train_records = make_sft_records(data, set(train_ids))
print('SFT examples:', len(train_records))
print(pd.DataFrame(train_records)[['event_id','condition','literacy_level']].value_counts().sort_index())

manifest = {
    'base_model': BASE_MODEL,
    'train_events': train_ids,
    'test_events': test_ids,
    'train_examples': len(train_records),
    'seed': SEED,
    'smoke_test': SMOKE_TEST,
}
(results_dir/'training_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

## 13. Tokenization and LoRA/QLoRA fine-tuning

The custom dataset masks the prompt portion and computes loss only on the assistant response. This avoids teaching the model to reproduce the user/system prompt tokens as the training target.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

class QwenSFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.items=[]
        self.tokenizer=tokenizer
        self.max_length=max_length
        for rec in records:
            msgs=rec['messages']
            # Prefix = system + user; full = system + user + assistant target.
            try:
                prefix = tokenizer.apply_chat_template(msgs[:-1], tokenize=True, add_generation_prompt=True, enable_thinking=False)
                full = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False, enable_thinking=False)
            except TypeError:
                prefix = tokenizer.apply_chat_template(msgs[:-1], tokenize=True, add_generation_prompt=True)
                full = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False)

            # Newer transformers/tokenizers versions may return a tokenizers.Encoding
            # from apply_chat_template(tokenize=True). The training code needs a plain
            # Python list of integer token IDs for padding and torch.tensor().
            if hasattr(prefix, 'ids'):
                prefix = prefix.ids
            elif hasattr(prefix, 'input_ids'):
                prefix = prefix.input_ids
            prefix = list(prefix)

            if hasattr(full, 'ids'):
                full = full.ids
            elif hasattr(full, 'input_ids'):
                full = full.input_ids
            full = list(full)

            full = full[:max_length]
            prefix_len = min(len(prefix), len(full))
            labels = [-100] * prefix_len + full[prefix_len:]
            self.items.append({'input_ids':full, 'labels':labels, 'event_id':rec['event_id']})

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    maxlen=max(len(x['input_ids']) for x in batch)
    pad=qwen_tokenizer.pad_token_id
    input_ids=[]; labels=[]; attn=[]
    for x in batch:
        n=maxlen-len(x['input_ids'])
        input_ids.append(x['input_ids']+[pad]*n)
        labels.append(x['labels']+[-100]*n)
        attn.append([1]*len(x['input_ids'])+[0]*n)
    return {
        'input_ids':torch.tensor(input_ids,dtype=torch.long),
        'labels':torch.tensor(labels,dtype=torch.long),
        'attention_mask':torch.tensor(attn,dtype=torch.long),
    }

train_ds=QwenSFTDataset(train_records, qwen_tokenizer, MAX_LENGTH)
train_loader=DataLoader(train_ds, batch_size=PER_DEVICE_BATCH, shuffle=True, collate_fn=collate)
print('Tokenized SFT rows:', len(train_ds))

if use_4bit_now:
    qwen_base = prepare_model_for_kbit_training(qwen_base)

lora_cfg=LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
)
qwen_base = get_peft_model(qwen_base, lora_cfg)
qwen_base.print_trainable_parameters()

### Compatibility note
Recent `transformers`/`tokenizers` versions can return a `tokenizers.Encoding` object from `apply_chat_template(..., tokenize=True)`. The dataset cell normalizes this to a plain list of token IDs before padding, preventing `TypeError: 'tokenizers.Encoding' object cannot be interpreted as an integer`.


## 14. Train

The default is intentionally modest: small batch, gradient accumulation, gradient checkpointing, and a short epoch count. Raise epochs only after inspecting the before/after behavior.

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

qwen_base.config.use_cache=False
try: qwen_base.gradient_checkpointing_enable()
except Exception: pass

optimizer=AdamW((p for p in qwen_base.parameters() if p.requires_grad), lr=LEARNING_RATE)
steps_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM)
total_steps=NUM_EPOCHS*steps_per_epoch
scheduler=get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1,total_steps//20), num_training_steps=total_steps)

qwen_base.train()
step=0
optimizer.zero_grad(set_to_none=True)
for epoch in range(NUM_EPOCHS):
    running=0.0
    for batch_i,batch in enumerate(train_loader):
        batch={k:v.to(qwen_base.device) for k,v in batch.items()}
        out=qwen_base(**batch)
        loss=out.loss/GRAD_ACCUM
        loss.backward()
        running += float(loss.detach())
        if (batch_i+1)%GRAD_ACCUM==0 or (batch_i+1)==len(train_loader):
            torch.nn.utils.clip_grad_norm_(qwen_base.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
            step += 1
            if step % 10 == 0 or step == 1:
                print(f'epoch={epoch+1}/{NUM_EPOCHS} step={step}/{total_steps} loss={running:.4f}')
                running=0.0

adapter_dir=ROOT/'models'/'qwen3-4b-lora'
adapter_dir.mkdir(parents=True, exist_ok=True)
qwen_base.save_pretrained(adapter_dir)
qwen_tokenizer.save_pretrained(adapter_dir)
print('Adapter saved to', adapter_dir)

## 15. Generate the local **after** responses

In [ ]:
qwen_base.eval()
after_records=[]
for i,r in eval_prompts.iterrows():
    if i % 5 == 0: print(f'Generating after: {i+1}/{len(eval_prompts)}')
    txt=generate_one(r['messages'])
    after_records.append({**r.to_dict(), 'Response':txt, 'ModelStage':'Local Qwen3-4B AFTER'})

after=pd.DataFrame(after_records)
after.to_json(results_dir/'local_after_raw.jsonl', orient='records', lines=True, force_ascii=False)
print('After generations:', len(after))

## 16. Score before and after

In [ ]:
# Reload paper-aligned evaluators after Qwen generation/training.
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Reloading BGE-M3...')
embedder = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
print('Reloading ModernCE-large-NLI...')
nli_model = CrossEncoder('dleemiller/ModernCE-large-nli', device=DEVICE, max_length=8192)
print('Reloading posture classifier...')
posture_pipe = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/bge-m3-zeroshot-v2.0',
    device=0 if DEVICE == 'cuda' else -1,
)
POSTURE_PIPE_AVAILABLE=True


In [ ]:
before_scored = evaluate_response_table(before, response_col='Response')
before_scored['ModelStage']='Local Qwen3-4B BEFORE'
after_scored = evaluate_response_table(after, response_col='Response')
after_scored['ModelStage']='Local Qwen3-4B AFTER'

scored=pd.concat([before_scored, after_scored], ignore_index=True)
scored.to_csv(results_dir/'before_after_scored.csv', index=False)
print('Scored rows:', len(scored))
print(scored.groupby(['ModelStage','Condition','Literacy_Level'])[['FK_Grade','Word_Count','ROUGE_Lsum_F1','Semantic_Loss_vs_High','NLI_Entailment']].mean().round(3))

## Optional: populate Specific Entity Count for new local generations


In [ ]:
# This block is skipped unless USE_GEMINI_ENTITY_EXTRACTOR=True and GEMINI_API_KEY is set.
if USE_GEMINI_ENTITY_EXTRACTOR:
    if not GEMINI_API_KEY:
        raise RuntimeError('USE_GEMINI_ENTITY_EXTRACTOR=True but GEMINI_API_KEY is missing.')
    import importlib.util
    from concurrent.futures import ThreadPoolExecutor, as_completed
    fact_path = ROOT/'original_relevant'/'factextract.py'
    spec = importlib.util.spec_from_file_location('supplied_factextract', fact_path)
    supplied_factextract = importlib.util.module_from_spec(spec)
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    spec.loader.exec_module(supplied_factextract)

    new_rows = pd.concat([
        before[['ID','Condition','Literacy_Level','Response']].assign(ModelStage='Local Qwen3-4B BEFORE'),
        after[['ID','Condition','Literacy_Level','Response']].assign(ModelStage='Local Qwen3-4B AFTER'),
    ], ignore_index=True)
    entity_records=[]
    def _extract(rec):
        e=supplied_factextract.extract_facts(str(rec['Response']))
        return {'ID':rec['ID'],'Condition':rec['Condition'],'Literacy_Level':rec['Literacy_Level'],'ModelStage':rec['ModelStage'],'Specific_Entity_Count':e['specific_count']}
    workers=min(5, max(1, os.cpu_count() or 1))
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs=[ex.submit(_extract, r) for _,r in new_rows.iterrows()]
        for j,f in enumerate(as_completed(futs),1):
            entity_records.append(f.result())
            if j%10==0: print('entity extraction:',j,'/',len(futs))
    ent=pd.DataFrame(entity_records)
    before_scored=before_scored.drop(columns=['Specific_Entity_Count'], errors='ignore').merge(ent[ent.ModelStage=='Local Qwen3-4B BEFORE'].drop(columns='ModelStage'), on=['ID','Condition','Literacy_Level'], how='left')
    after_scored=after_scored.drop(columns=['Specific_Entity_Count'], errors='ignore').merge(ent[ent.ModelStage=='Local Qwen3-4B AFTER'].drop(columns='ModelStage'), on=['ID','Condition','Literacy_Level'], how='left')
    scored=pd.concat([before_scored,after_scored],ignore_index=True)
    print('Entity counts populated.')
else:
    print('Specific Entity Count remains NaN for local generations. Historical Qwen3.6-27B counts come from the supplied extraction TSVs.')


## 17. Core before/after summary

For each metric, this shows mean score by condition and literacy level. The main comparison is local **BEFORE vs AFTER**.

In [ ]:
metric_cols=['FK_Grade','Word_Count','ROUGE_Lsum_F1','Specific_Entity_Count','Semantic_Loss_vs_High','NLI_Entailment','NLI_Neutral','NLI_Contradiction','Info_Comprehensive_Prob','Style_Formal_Prob']
summary=scored.groupby(['ModelStage','Condition','Literacy_Level'])[metric_cols].mean().reset_index()
summary.to_csv(results_dir/'before_after_summary.csv', index=False)
summary.round(3)

## 18. Paired High→Low statistics

These are descriptive paired event-level tests on the held-out set. The paper uses one-tailed paired tests by event ID with effect sizes. Here we compute the same style of comparison for the local before/after scores.

In [ ]:
from scipy.stats import ttest_rel

def paired_high_low(df, metric):
    rows=[]
    for stage in ['Local Qwen3-4B BEFORE','Local Qwen3-4B AFTER']:
        for condition in ['Explicit','Implicit']:
            s=df[(df.ModelStage==stage)&(df.Condition==condition)]
            piv=s.pivot_table(index='ID',columns='Literacy_Level',values=metric,aggfunc='first')
            if not set(['High Literacy','Low Literacy']).issubset(piv.columns): continue
            x=piv['High Literacy']; y=piv['Low Literacy']; keep=x.notna()&y.notna(); x=x[keep]; y=y[keep]
            # We test H > L for preservation-type metrics; for Semantic Loss we test H < L.
            if metric == 'Semantic_Loss_vs_High':
                t,p=ttest_rel(x,y,alternative='less'); diff=y-x
            else:
                t,p=ttest_rel(x,y,alternative='greater'); diff=x-y
            sd=diff.std(ddof=1); d=diff.mean()/sd if sd else np.nan
            rows.append({'ModelStage':stage,'Condition':condition,'Metric':metric,'n':len(diff),'High_mean':x.mean(),'Low_mean':y.mean(),'High_minus_Low':(x-y).mean(),'t':t,'p_one_tailed':p,'Cohen_d_paired':d})
    return pd.DataFrame(rows)

stats=pd.concat([paired_high_low(scored,m) for m in metric_cols],ignore_index=True)
stats.to_csv(results_dir/'paired_statistics.csv',index=False)
stats.round(4)

## 19. Before → after gap plots

These plots deliberately keep metrics separate. The paper does not define a single composite “quality” score, so the notebook does not invent one as a headline result.

In [ ]:
# Direction-aware gap definitions: positive is a larger High-vs-Low separation.
def mean_high_low(stage, metric, condition):
    s=scored[(scored.ModelStage==stage)&(scored.Condition==condition)]
    piv=s.pivot_table(index='ID',columns='Literacy_Level',values=metric,aggfunc='first')
    if not {'High Literacy','Low Literacy'}.issubset(piv.columns): return np.nan
    return (piv['High Literacy']-piv['Low Literacy']).mean()

for metric in ['ROUGE_Lsum_F1','Specific_Entity_Count','Semantic_Loss_vs_High','NLI_Entailment','Info_Comprehensive_Prob','Style_Formal_Prob','FK_Grade','Word_Count']:
    fig,ax=plt.subplots(figsize=(8,5))
    conditions=['Explicit','Implicit']
    before_vals=[mean_high_low('Local Qwen3-4B BEFORE',metric,c) for c in conditions]
    after_vals=[mean_high_low('Local Qwen3-4B AFTER',metric,c) for c in conditions]

    x=np.arange(len(conditions))
    width=0.36
    ax.bar(x - width/2, before_vals, width, label='Before')
    ax.bar(x + width/2, after_vals, width, label='After')
    ax.set_xticks(x, conditions)
    ax.set_title(f'High − Low mean gap: {metric}')
    ax.set_ylabel('Mean gap')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    fig.savefig(fig_dir/f'gap_{metric}.png',dpi=180)
    plt.show()


## 20. Direct before vs after comparison table

In [ ]:
# Compact table: change in High-Low gap (After - Before)
rows=[]
for metric in metric_cols:
    for condition in ['Explicit','Implicit']:
        b=mean_high_low('Local Qwen3-4B BEFORE',metric,condition)
        a=mean_high_low('Local Qwen3-4B AFTER',metric,condition)
        rows.append({'Metric':metric,'Condition':condition,'Before_Gap':b,'After_Gap':a,'After_minus_Before':a-b})
gap_df=pd.DataFrame(rows)
gap_df.to_csv(results_dir/'before_after_gaps.csv',index=False)
gap_df.round(4)

## 21. Example response inspection

Use this cell to inspect the same event before and after training. This is often more informative than a single aggregate score when an evaluator moves.

In [ ]:
example_id=test_ids[0]
examples=pd.concat([
    before[(before.ID==example_id)][['Condition','Literacy_Level','Eval_User_Input','Response']].assign(Stage='BEFORE'),
    after[(after.ID==example_id)][['Condition','Literacy_Level','Eval_User_Input','Response']].assign(Stage='AFTER')
],ignore_index=True)
for r in examples.itertuples():
    print('\n'+'='*100)
    print(r.Stage, '|', r.Condition, '|', r.Literacy_Level)
    print('PROMPT:\n', r.Eval_User_Input)
    print('\nRESPONSE:\n', r.Response[:5000])


## 22. Optional: Specific Entity Count on new local responses

The original supplied pipeline uses a Gemini model at temperature 0 for the 17-category structured extractor. Keep this optional because it requires an external API credential. The historical Qwen counts are already available in the supplied extraction TSVs.

To enable this, adapt the included `original_relevant/factextract.py` to point at `local_before_raw.jsonl` and `local_after_raw.jsonl`, then merge its output into `before_after_scored.csv` using `ID + Condition + Literacy_Level`.

## 23. Final report

The final markdown report records the split, settings, the before/after summary, and the High-Low gap change. Interpret the direction of each metric separately; no composite score is assumed.

In [ ]:
report=[]
report.append('# Qwen Before/After Literacy-Stratification Audit')
report.append('')
report.append(f'- Base model: `{BASE_MODEL}`')
report.append(f'- Training events: {len(train_ids)}')
report.append(f'- Held-out test events: {len(test_ids)}')
report.append(f'- Smoke test: `{SMOKE_TEST}`')
report.append('')
report.append('## Before/After mean scores')
report.append('')
report.append(summary.round(4).to_markdown(index=False))
report.append('')
report.append('## High-Low gap change (After - Before)')
report.append('')
report.append(gap_df.round(4).to_markdown(index=False))
report.append('')
report.append('## Interpretation rule')
report.append('')
report.append('- FK grade and word count primarily document register/length adaptation.')
report.append('- ROUGE, NLI, semantic loss, entity count and posture are interpreted together for substantive/service asymmetry.')
report.append('- A post-training change is an empirical outcome, not an assumed improvement.')
report.append('- The historical Qwen3.6-27B is contextual; the causal-style before/after comparison is local base Qwen3-4B vs the same model after fine-tuning.')
(results_dir/'BEFORE_AFTER_REPORT.md').write_text('\n'.join(report),encoding='utf-8')
print('Saved:', results_dir/'BEFORE_AFTER_REPORT.md')

## 24. Output checklist

After the full run, the most important files are:

- `results/before_after_scored.csv`
- `results/before_after_summary.csv`
- `results/paired_statistics.csv`
- `results/before_after_gaps.csv`
- `results/BEFORE_AFTER_REPORT.md`
- `models/qwen3-4b-lora/`